In [1]:
import os
import dgl
import numpy as np
import torch
from sklearn.metrics import pairwise_distances
from scipy.linalg import inv

In [ ]:

chromosomes = range(1, 23)


MAHALANOBIS_THRESHOLD = 1  

output_dir = 'SDGL'
os.makedirs(output_dir, exist_ok=True)

for i in chromosomes:
    print(f"\nProcessing chromosome {i}...")
    

    original_graph_path = f"DGL/chr{i}.dgl"
    g, _ = dgl.load_graphs(original_graph_path)
    g = g[0]  
    
    num_nodes = g.number_of_nodes()
    

    # g_ATAC = np.loadtxt(f'ATAC/chr{i}.txt')
    # g_RNA = np.loadtxt(f'RNA/chr{i}.txt')
    # g_Pol2 = np.loadtxt(f'RNApol2/chr{i}.txt')
    # g_CTCF = np.loadtxt(f'CTCF/chr{i}.txt')
    # g_H3K27ac = np.loadtxt(f'H3K27ac/chr{i}.txt')
    

    # g_feats = np.vstack((g_ATAC, g_RNA, g_Pol2, g_CTCF, g_H3K27ac)).T  # shape: (num_nodes, num_features)
    
    g_feats = np.loadtxt('feats/chr' + str(i) + '_features.txt', delimiter='\t')


    print("Calculating covariance matrix inverse for Mahalanobis distance...")
    cov_matrix = np.cov(g_feats, rowvar=False)
    try:
        inv_cov_matrix = inv(cov_matrix)
    except np.linalg.LinAlgError:
        inv_cov_matrix = np.linalg.pinv(cov_matrix)

    print("Calculating Mahalanobis distance matrix...")
    distance_matrix = pairwise_distances(g_feats, metric='mahalanobis', VI=inv_cov_matrix)

    similarity_matrix = 1 / (1 + distance_matrix)
    

    similarity_mask = similarity_matrix >= MAHALANOBIS_THRESHOLD
    np.fill_diagonal(similarity_mask, False)  
    num_edges = np.sum(similarity_mask)
    print(f"Number of edges with similarity >= {MAHALANOBIS_THRESHOLD}: {num_edges}")

    original_num_edges = g.number_of_edges()
    edge_ratio = num_edges / original_num_edges if original_num_edges > 0 else float('inf')
    print(f"Edge ratio compared to original graph: {edge_ratio:.2f}")
    

    print("Applying Mahalanobis similarity threshold to create edges...")
    src, dst = np.where(similarity_mask)
    
    similarities = similarity_matrix[src, dst]
    
    print("Creating new DGL graph...")
    new_g = dgl.graph((src, dst), num_nodes=num_nodes)
    

    new_g.edata['edge_feature'] = torch.tensor(similarities, dtype=torch.float32)
    

    new_g.ndata['label'] = g.ndata['label'].clone()
    

    new_graph_path = os.path.join(output_dir, f"chr{i}.dgl")
    dgl.save_graphs(new_graph_path, [new_g])
    
    print(f"Chromosome {i} processed and saved to {new_graph_path}.\n")
    



Processing chromosome 1...
Calculating covariance matrix inverse for Mahalanobis distance...
Calculating Mahalanobis distance matrix...
Number of edges with similarity >= 1: 7787322
Edge ratio compared to original graph: 6.11
Applying Mahalanobis similarity threshold to create edges...
Creating new DGL graph...
Chromosome 1 processed and saved to SDGL\chr1.dgl.


Processing chromosome 2...
Calculating covariance matrix inverse for Mahalanobis distance...
Calculating Mahalanobis distance matrix...
Number of edges with similarity >= 1: 510716
Edge ratio compared to original graph: 0.36
Applying Mahalanobis similarity threshold to create edges...
Creating new DGL graph...
Chromosome 2 processed and saved to SDGL\chr2.dgl.


Processing chromosome 3...
Calculating covariance matrix inverse for Mahalanobis distance...
Calculating Mahalanobis distance matrix...
Number of edges with similarity >= 1: 111294
Edge ratio compared to original graph: 0.09
Applying Mahalanobis similarity threshold t